## 1) Configuration

In [1]:
from pathlib import Path
import pandas as pd

PARENT_DIR = Path.cwd().parent

DATASET_CONFIG = {
    "chiffchaff": {
        "allowed_splits": {"withinyear", "acrossyear"},
    },
    "pipit": {
        "allowed_splits": {"withinyear", "acrossyear"},
    },
    "littleowl": {
        "allowed_splits": {"acrossyear"},
    },
    "littlepenguin": {
        "allowed_splits": {"withinyear"},
    },
    "kiwi": {
        "allowed_splits": {"acrossyear"},
    },
    "rtbc": {
        "allowed_splits": {"acrossyear"},
    },
    # If later you want to include great tit:
    "greatTit": {
        "allowed_splits": {"acrossyear"},
    },
}

RESULTS_ROOT = PARENT_DIR / "Results"

REQUIRED_COLS = [
    "run_name",
    "seed",
    "dataset_name",
    "split_type",
    "sequence_mode",
    "embedding_dim",
    "test_accuracy",
    "test_f1_macro",
    "test_roc_auc_macro",
]

print("RESULTS_ROOT:", RESULTS_ROOT)

RESULTS_ROOT: /teamspace/studios/this_studio/Results


## 2) Find all files with metrics

In [ ]:
metric_files = []

for dataset_name, cfg in DATASET_CONFIG.items():
    for split_type in sorted(cfg["allowed_splits"]):
        metrics_dir = RESULTS_ROOT / dataset_name / split_type / "Metrics"
        if not metrics_dir.exists():
            continue

        # Only the per-run files, not aggregate summaries
        files = sorted(metrics_dir.glob("*_final_metrics.csv"))
        metric_files.extend(files)

print(f"Found {len(metric_files)} metric files.")
for f in metric_files[:len(metric_files)]:
    print(" -", f)

Found 52 metric files.
 - /teamspace/studios/this_studio/Results/chiffchaff/acrossyear/Metrics/chiffchaff_acrossyear_birdnet_gap_final_metrics.csv
 - /teamspace/studios/this_studio/Results/chiffchaff/acrossyear/Metrics/chiffchaff_acrossyear_birdnet_lstm_ordered_final_metrics.csv
 - /teamspace/studios/this_studio/Results/chiffchaff/acrossyear/Metrics/chiffchaff_acrossyear_birdnet_lstm_shuffled_final_metrics.csv
 - /teamspace/studios/this_studio/Results/chiffchaff/acrossyear/Metrics/chiffchaff_acrossyear_birdnet_onset_final_metrics.csv
 - /teamspace/studios/this_studio/Results/chiffchaff/acrossyear/Metrics/chiffchaff_acrossyear_perch_gap_final_metrics.csv
 - /teamspace/studios/this_studio/Results/chiffchaff/acrossyear/Metrics/chiffchaff_acrossyear_perch_onset_final_metrics.csv
 - /teamspace/studios/this_studio/Results/chiffchaff/withinyear/Metrics/chiffchaff_withinyear_birdnet_gap_final_metrics.csv
 - /teamspace/studios/this_studio/Results/chiffchaff/withinyear/Metrics/chiffchaff_withiny

## 3) load everithing in a single Dataframe

In [ ]:
all_metrics = []

for file_path in metric_files:
    try:
        df = pd.read_csv(file_path)

        missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
        if missing_cols:
            print(f"Skipping {file_path.name}: missing columns {missing_cols}")
            continue

        # Add file provenance
        df["metrics_file"] = file_path.name
        df["metrics_path"] = str(file_path)

        # If needed, infer dataset/split from path as a fallback
        parts = file_path.parts
        # .../Results/<dataset>/<split>/Metrics/<file>
        try:
            results_idx = parts.index("Results")
            df["dataset_from_path"] = parts[results_idx + 1]
            df["split_from_path"] = parts[results_idx + 2]
        except ValueError:
            df["dataset_from_path"] = None
            df["split_from_path"] = None

        all_metrics.append(df)

    except Exception as e:
        print(f"Error reading {file_path}: {e}")

if not all_metrics:
    raise ValueError("No valid metrics files were loaded.")

combined_metrics_df = pd.concat(all_metrics, ignore_index=True)

# Optional: remove duplicates by run_name + seed, keeping the last occurrence
combined_metrics_df = (
    combined_metrics_df
    .drop_duplicates(subset=["run_name", "seed"], keep="last")
    .sort_values(by=["dataset_name", "split_type", "sequence_mode", "seed"])
    .reset_index(drop=True)
)

print(f"Combined rows: {len(combined_metrics_df)}")
display(combined_metrics_df.head())
display(combined_metrics_df.tail())

Skipping chiffchaff_acrossyear_birdnet_gap_final_metrics.csv: missing columns ['sequence_mode']
Skipping chiffchaff_acrossyear_birdnet_onset_final_metrics.csv: missing columns ['sequence_mode']
Skipping chiffchaff_acrossyear_perch_gap_final_metrics.csv: missing columns ['sequence_mode']
Skipping chiffchaff_acrossyear_perch_onset_final_metrics.csv: missing columns ['sequence_mode']
Skipping chiffchaff_withinyear_birdnet_gap_final_metrics.csv: missing columns ['sequence_mode']
Skipping chiffchaff_withinyear_birdnet_onset_final_metrics.csv: missing columns ['sequence_mode']
Skipping chiffchaff_withinyear_perch_gap_final_metrics.csv: missing columns ['sequence_mode']
Skipping chiffchaff_withinyear_perch_onset_final_metrics.csv: missing columns ['sequence_mode']
Skipping pipit_acrossyear_birdnet_gap_final_metrics.csv: missing columns ['sequence_mode']
Skipping pipit_acrossyear_birdnet_onset_final_metrics.csv: missing columns ['sequence_mode']
Skipping pipit_acrossyear_perch_gap_final_metric

,run_name,seed,dataset_name,split_type,sequence_mode,embedding_dim,hidden_dim,learning_rate,val_accuracy,val_balanced_accuracy,...,test_f1_weighted,test_roc_auc_macro,metrics_file,metrics_path,dataset_from_path,split_from_path,n_frames_test,split_mode,test_cmc1,test_map5
0,chiffchaff_acrossyear_birdnet_lstm_ordered,18,chiffchaff,acrossyear,ordered,1024,256,0.001,0.980952,0.983974,...,0.980939,0.999588,chiffchaff_acrossyear_birdnet_lstm_ordered_fin...,/teamspace/studios/this_studio/Results/chiffch...,chiffchaff,acrossyear,NaN,NaN,NaN,NaN
1,chiffchaff_acrossyear_birdnet_lstm_ordered,23,chiffchaff,acrossyear,ordered,1024,256,0.001,1.000000,1.000000,...,0.980976,0.999742,chiffchaff_acrossyear_birdnet_lstm_ordered_fin...,/teamspace/studios/this_studio/Results/chiffch...,chiffchaff,acrossyear,NaN,NaN,NaN,NaN
2,chiffchaff_acrossyear_birdnet_lstm_ordered,46,chiffchaff,acrossyear,ordered,1024,256,0.001,0.990476,0.991667,...,0.886394,0.997707,chiffchaff_acrossyear_birdnet_lstm_ordered_fin...,/teamspace/studios/this_studio/Results/chiffch...,chiffchaff,acrossyear,NaN,NaN,NaN,NaN
3,chiffchaff_acrossyear_birdnet_lstm_ordered,123,chiffchaff,acrossyear,ordered,1024,256,0.001,0.990476,0.985714,...,0.980952,0.999736,chiffchaff_acrossyear_birdnet_lstm_ordered_fin...,/teamspace/studios/this_studio/Results/chiffch...,chiffchaff,acrossyear,NaN,NaN,NaN,NaN
4,chiffchaff_acrossyear_birdnet_lstm_ordered,321,chiffchaff,acrossyear,ordered,1024,256,0.001,0.971429,0.969246,...,0.980859,0.999871,chiffchaff_acrossyear_birdnet_lstm_ordered_fin...,/teamspace/studios/this_studio/Results/chiffch...,chiffchaff,acrossyear,NaN,NaN,NaN,NaN


,run_name,seed,dataset_name,split_type,sequence_mode,embedding_dim,hidden_dim,learning_rate,val_accuracy,val_balanced_accuracy,...,test_f1_weighted,test_roc_auc_macro,metrics_file,metrics_path,dataset_from_path,split_from_path,n_frames_test,split_mode,test_cmc1,test_map5
74,rtbc_acrossyear_birdnet_lstm_ordered,23,rtbc,acrossyear,ordered,1024,256,0.001,0.949580,0.938794,...,0.963578,0.998292,rtbc_acrossyear_birdnet_lstm_ordered_final_met...,/teamspace/studios/this_studio/Results/rtbc/ac...,rtbc,acrossyear,NaN,NaN,NaN,NaN
75,rtbc_acrossyear_birdnet_lstm_ordered,42,rtbc,acrossyear,ordered,1024,256,0.001,0.971989,0.960644,...,0.960481,0.999277,rtbc_acrossyear_birdnet_lstm_ordered_final_met...,/teamspace/studios/this_studio/Results/rtbc/ac...,rtbc,acrossyear,NaN,NaN,NaN,NaN
76,rtbc_acrossyear_birdnet_lstm_ordered,46,rtbc,acrossyear,ordered,1024,256,0.001,0.969188,0.966691,...,0.965969,0.999642,rtbc_acrossyear_birdnet_lstm_ordered_final_met...,/teamspace/studios/this_studio/Results/rtbc/ac...,rtbc,acrossyear,NaN,NaN,NaN,NaN
77,rtbc_acrossyear_birdnet_lstm_ordered,123,rtbc,acrossyear,ordered,1024,256,0.001,0.991597,0.978886,...,0.974658,0.999780,rtbc_acrossyear_birdnet_lstm_ordered_final_met...,/teamspace/studios/this_studio/Results/rtbc/ac...,rtbc,acrossyear,NaN,NaN,NaN,NaN
78,rtbc_acrossyear_birdnet_lstm_ordered,321,rtbc,acrossyear,ordered,1024,256,0.001,0.971989,0.978779,...,0.963271,0.998446,rtbc_acrossyear_birdnet_lstm_ordered_final_met...,/teamspace/studios/this_studio/Results/rtbc/ac...,rtbc,acrossyear,NaN,NaN,NaN,NaN


## 4) Save unified dataframe

In [4]:
OUTPUT_DIR = PARENT_DIR / "Results" / "_combined_lstm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_METRICS_FILE = OUTPUT_DIR / "all_metrics_lstm_combined.csv"
combined_metrics_df.to_csv(COMBINED_METRICS_FILE, index=False)

print(f"Saved combined metrics to: {COMBINED_METRICS_FILE}")

# COMBINED_METRICS_FILE_XLSX = OUTPUT_DIR / "all_metrics_combined.xlsx"
# combined_metrics_df.to_excel(COMBINED_METRICS_FILE_XLSX, index=False)

# print(f"Saved combined metrics to: {COMBINED_METRICS_FILE_XLSX}")

Saved combined metrics to: /teamspace/studios/this_studio/Results/_combined_lstm/all_metrics_lstm_combined.csv


## 5) Aggregate summary per dataset/split/sequence

In [ ]:
summary_df = (
    combined_metrics_df
    .groupby(["dataset_name", "split_type", "sequence_mode", "embedding_dim"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        test_accuracy_mean=("test_accuracy", "mean"),
        test_accuracy_sd=("test_accuracy", lambda x: x.std(ddof=1)),
        test_f1_macro_mean=("test_f1_macro", "mean"),
        test_f1_macro_sd=("test_f1_macro", lambda x: x.std(ddof=1)),
        test_roc_auc_macro_mean=("test_roc_auc_macro", "mean"),
        test_roc_auc_macro_sd=("test_roc_auc_macro", lambda x: x.std(ddof=1)),
        test_cmc1_mean=("test_cmc1", "mean") if "test_cmc1" in combined_metrics_df.columns else pd.Series(dtype=float),
        test_cmc1_sd=("test_cmc1", lambda x: x.std(ddof=1)) if "test_cmc1" in combined_metrics_df.columns else pd.Series(dtype=float),
        test_map5_mean=("test_map5", "mean") if "test_map5" in combined_metrics_df.columns else pd.Series(dtype=float), 
        test_map5_sd=("test_map5", lambda x: x.std(ddof=1)) if "test_map5" in combined_metrics_df.columns else pd.Series(dtype=float),
    )
)

display(summary_df)

,dataset_name,split_type,sequence_mode,embedding_dim,n_seeds,test_accuracy_mean,test_accuracy_sd,test_f1_macro_mean,test_f1_macro_sd,test_roc_auc_macro_mean,test_roc_auc_macro_sd,test_cmc1_mean,test_cmc1_sd,test_map5_mean,test_map5_sd
0,chiffchaff,acrossyear,ordered,1024,5,0.963810,0.038333,0.964094,0.036709,0.999329,0.000912,NaN,NaN,NaN,NaN
1,chiffchaff,acrossyear,shuffled,1024,5,0.950476,0.035888,0.953123,0.032343,0.998726,0.001623,NaN,NaN,NaN,NaN
2,chiffchaff,withinyear,ordered,1024,5,0.974679,0.007635,0.967293,0.009826,0.999602,0.000238,NaN,NaN,NaN,NaN
3,chiffchaff,withinyear,shuffled,1024,5,0.974359,0.003884,0.967605,0.003864,0.999547,0.000167,NaN,NaN,NaN,NaN
4,greatTit,acrossyear,ordered,1024,5,0.968918,0.001755,0.939932,0.005214,0.999777,0.000121,0.968918,0.001755,0.980019,0.001141
5,greatTit,acrossyear,shuffled,1024,5,0.966825,0.003446,0.937028,0.004528,0.999804,0.000063,0.966825,0.003446,0.978616,0.002133
6,kiwi,acrossyear,ordered,1024,5,0.938462,0.016666,0.951061,0.011925,0.998244,0.002792,NaN,NaN,NaN,NaN
7,kiwi,acrossyear,shuffled,1024,5,0.916484,0.038540,0.924611,0.038439,0.997960,0.002188,NaN,NaN,NaN,NaN
8,littleowl,acrossyear,ordered,1024,5,0.983246,0.011939,0.983373,0.012855,0.999768,0.000271,NaN,NaN,NaN,NaN
9,littleowl,acrossyear,shuffled,1024,3,0.986038,0.003023,0.986892,0.002362,0.999715,0.000347,NaN,NaN,NaN,NaN


## 6) pretty summary (mean ± SD)

In [6]:
pretty_summary_df = summary_df.copy()

pretty_summary_df["test_accuracy"] = pretty_summary_df.apply(
    lambda r: f"{r['test_accuracy_mean']:.4f} ± {r['test_accuracy_sd']:.4f}", axis=1
)
pretty_summary_df["test_f1_macro"] = pretty_summary_df.apply(
    lambda r: f"{r['test_f1_macro_mean']:.4f} ± {r['test_f1_macro_sd']:.4f}", axis=1
)
pretty_summary_df["test_roc_auc_macro"] = pretty_summary_df.apply(
    lambda r: f"{r['test_roc_auc_macro_mean']:.4f} ± {r['test_roc_auc_macro_sd']:.4f}", axis=1
)

pretty_summary_df["test_cmc1"] = pretty_summary_df.apply(
    lambda r: f"{r['test_cmc1_mean']:.4f} ± {r['test_cmc1_sd']:.4f}" if "test_cmc1_mean" in r else "N/A", axis=1
)

pretty_summary_df["test_map5"] = pretty_summary_df.apply(
    lambda r: f"{r['test_map5_mean']:.4f} ± {r['test_map5_sd']:.4f}" if "test_map5_mean" in r else "N/A", axis=1
)

pretty_summary_df = pretty_summary_df[
    [
        "dataset_name",
        "split_type",
        "sequence_mode",
        "embedding_dim",
        "n_seeds",
        "test_accuracy",
        "test_f1_macro",
        "test_roc_auc_macro",
        "test_cmc1",
        "test_map5",
    ]
]

display(pretty_summary_df)


,dataset_name,split_type,sequence_mode,embedding_dim,n_seeds,test_accuracy,test_f1_macro,test_roc_auc_macro,test_cmc1,test_map5
0,chiffchaff,acrossyear,ordered,1024,5,0.9638 ± 0.0383,0.9641 ± 0.0367,0.9993 ± 0.0009,nan ± nan,nan ± nan
1,chiffchaff,acrossyear,shuffled,1024,5,0.9505 ± 0.0359,0.9531 ± 0.0323,0.9987 ± 0.0016,nan ± nan,nan ± nan
2,chiffchaff,withinyear,ordered,1024,5,0.9747 ± 0.0076,0.9673 ± 0.0098,0.9996 ± 0.0002,nan ± nan,nan ± nan
3,chiffchaff,withinyear,shuffled,1024,5,0.9744 ± 0.0039,0.9676 ± 0.0039,0.9995 ± 0.0002,nan ± nan,nan ± nan
4,greatTit,acrossyear,ordered,1024,5,0.9689 ± 0.0018,0.9399 ± 0.0052,0.9998 ± 0.0001,0.9689 ± 0.0018,0.9800 ± 0.0011
5,greatTit,acrossyear,shuffled,1024,5,0.9668 ± 0.0034,0.9370 ± 0.0045,0.9998 ± 0.0001,0.9668 ± 0.0034,0.9786 ± 0.0021
6,kiwi,acrossyear,ordered,1024,5,0.9385 ± 0.0167,0.9511 ± 0.0119,0.9982 ± 0.0028,nan ± nan,nan ± nan
7,kiwi,acrossyear,shuffled,1024,5,0.9165 ± 0.0385,0.9246 ± 0.0384,0.9980 ± 0.0022,nan ± nan,nan ± nan
8,littleowl,acrossyear,ordered,1024,5,0.9832 ± 0.0119,0.9834 ± 0.0129,0.9998 ± 0.0003,nan ± nan,nan ± nan
9,littleowl,acrossyear,shuffled,1024,3,0.9860 ± 0.0030,0.9869 ± 0.0024,0.9997 ± 0.0003,nan ± nan,nan ± nan


## 7) save summaries

In [7]:

SUMMARY_FILE = OUTPUT_DIR / "all_metrics_lstm_summary_numeric.csv"
PRETTY_SUMMARY_METRICS_FILE = OUTPUT_DIR / "all_metrics_lstm_summary_pretty.csv" 

summary_df.to_csv(SUMMARY_FILE, index=False)
pretty_summary_df.to_csv(PRETTY_SUMMARY_METRICS_FILE, index=False)

print(f"Saved numeric summary to: {SUMMARY_FILE}")
print(f"Saved combined metrics to: {PRETTY_SUMMARY_METRICS_FILE}")

# COMBINED_METRICS_FILE_XLSX = OUTPUT_DIR / "all_metrics_combined.xlsx"
# combined_metrics_df.to_excel(COMBINED_METRICS_FILE_XLSX, index=False)

# print(f"Saved combined metrics to: {COMBINED_METRICS_FILE_XLSX}")

Saved numeric summary to: /teamspace/studios/this_studio/Results/_combined_lstm/all_metrics_lstm_summary_numeric.csv
Saved combined metrics to: /teamspace/studios/this_studio/Results/_combined_lstm/all_metrics_lstm_summary_pretty.csv


## 8) pivot table for quick comparison

### f1 macro

In [8]:
pivot_df = pretty_summary_df.pivot_table(
    index=["dataset_name", "split_type"],
    columns=["sequence_mode"],
    values="test_f1_macro",
    aggfunc="first"
)

display(pivot_df)

sequence_mode                     ordered         shuffled
dataset_name  split_type                                  
chiffchaff    acrossyear  0.9641 ± 0.0367  0.9531 ± 0.0323
              withinyear  0.9673 ± 0.0098  0.9676 ± 0.0039
greatTit      acrossyear  0.9399 ± 0.0052  0.9370 ± 0.0045
kiwi          acrossyear  0.9511 ± 0.0119  0.9246 ± 0.0384
littleowl     acrossyear  0.9834 ± 0.0129  0.9869 ± 0.0024
littlepenguin withinyear  0.9323 ± 0.0086              NaN
pipit         acrossyear  0.9518 ± 0.0338  0.9509 ± 0.0130
              withinyear  0.9694 ± 0.0261  0.9537 ± 0.0262
rtbc          acrossyear  0.9588 ± 0.0116              NaN

### accuracy

In [9]:
pivot_df = pretty_summary_df.pivot_table(
    index=["dataset_name", "split_type"],
    columns=["sequence_mode"],
    values="test_accuracy",
    aggfunc="first"
)

display(pivot_df)

sequence_mode                     ordered         shuffled
dataset_name  split_type                                  
chiffchaff    acrossyear  0.9638 ± 0.0383  0.9505 ± 0.0359
              withinyear  0.9747 ± 0.0076  0.9744 ± 0.0039
greatTit      acrossyear  0.9689 ± 0.0018  0.9668 ± 0.0034
kiwi          acrossyear  0.9385 ± 0.0167  0.9165 ± 0.0385
littleowl     acrossyear  0.9832 ± 0.0119  0.9860 ± 0.0030
littlepenguin withinyear  0.9461 ± 0.0084              NaN
pipit         acrossyear  0.9545 ± 0.0311  0.9531 ± 0.0133
              withinyear  0.9706 ± 0.0239  0.9580 ± 0.0210
rtbc          acrossyear  0.9659 ± 0.0048              NaN

### ROC_AUC

In [10]:
pivot_df = pretty_summary_df.pivot_table(
    index=["dataset_name", "split_type"],
    columns=["sequence_mode"],
    values="test_roc_auc_macro",
    aggfunc="first"
)

display(pivot_df)

sequence_mode                     ordered         shuffled
dataset_name  split_type                                  
chiffchaff    acrossyear  0.9993 ± 0.0009  0.9987 ± 0.0016
              withinyear  0.9996 ± 0.0002  0.9995 ± 0.0002
greatTit      acrossyear  0.9998 ± 0.0001  0.9998 ± 0.0001
kiwi          acrossyear  0.9982 ± 0.0028  0.9980 ± 0.0022
littleowl     acrossyear  0.9998 ± 0.0003  0.9997 ± 0.0003
littlepenguin withinyear  0.9986 ± 0.0006              NaN
pipit         acrossyear  0.9969 ± 0.0029  0.9975 ± 0.0015
              withinyear  0.9995 ± 0.0008  0.9987 ± 0.0008
rtbc          acrossyear  0.9992 ± 0.0006              NaN

### CMC1

In [11]:
pivot_df = pretty_summary_df.pivot_table(
    index=["dataset_name", "split_type"],
    columns=["sequence_mode"],
    values="test_cmc1",
    aggfunc="first"
)

display(pivot_df)

sequence_mode                     ordered         shuffled
dataset_name  split_type                                  
chiffchaff    acrossyear        nan ± nan        nan ± nan
              withinyear        nan ± nan        nan ± nan
greatTit      acrossyear  0.9689 ± 0.0018  0.9668 ± 0.0034
kiwi          acrossyear        nan ± nan        nan ± nan
littleowl     acrossyear        nan ± nan        nan ± nan
littlepenguin withinyear        nan ± nan              NaN
pipit         acrossyear        nan ± nan        nan ± nan
              withinyear        nan ± nan        nan ± nan
rtbc          acrossyear        nan ± nan              NaN

## MAP5

In [12]:
pivot_df = pretty_summary_df.pivot_table(
    index=["dataset_name", "split_type"],
    columns=["sequence_mode"],
    values="test_map5",
    aggfunc="first"
)

display(pivot_df)

sequence_mode                     ordered         shuffled
dataset_name  split_type                                  
chiffchaff    acrossyear        nan ± nan        nan ± nan
              withinyear        nan ± nan        nan ± nan
greatTit      acrossyear  0.9800 ± 0.0011  0.9786 ± 0.0021
kiwi          acrossyear        nan ± nan        nan ± nan
littleowl     acrossyear        nan ± nan        nan ± nan
littlepenguin withinyear        nan ± nan              NaN
pipit         acrossyear        nan ± nan        nan ± nan
              withinyear        nan ± nan        nan ± nan
rtbc          acrossyear        nan ± nan              NaN